In [ ]:
from google.colab import files
files.upload()


Saving train_data.zip to train_data (1).zip


{'train_data (1).zip': b'PK\x03\x04\x14\x00\x00\x00\x08\x00\x91\xb94\\/\xeb\x17*\xd1\x03\x00\x00K\xf1\x00\x00\x15\x00\x1c\x00data/train_tokens.txtUT\t\x00\x03\x89\xbeoi\x9b\xbeoiux\x0b\x00\x01\x04\xe8\x03\x00\x00\x04\xe8\x03\x00\x00\xed\xd9\xddR\xda@\x14\xc0\xf1\xfb>E^\x80\xf1\x05l\x1d\xab\xd8q\x86V\xfc\xeaL\xaf\x9c5Y\xc9\x8e\xf9r\xb3\x81\xe1\xed\xbb\xa0\xb5\x95\x82n\x96\x98\x80\xfco@\x90\xb3\xe7\xfc6\'\xd9%\xa4\xd2\xc4yt\xf3\xad\x7f\x15\x14\xc2\xc47\xc1^\xa0ei\xec\x93\x88R\x95\xcd\x9e\x8b"Q\xa10*\xcfzc\xa9K\xfb\x1c\x9c_\xf7/~\x05\x9f\x83\x13\x91\x94\xf2S\xbar\x8c\xaa\x94\xda>M\xe2\\\xa4\xca9jY\xe60\xcf\xee\xd4\xa8\xd2\xf3Wk\x8d\xf4\x11\x0c~#%"\x1bUb$K\xe7\x88B\xe7Q\x15\x9a\xd2\xfeYJ\xa1\xc388\x08\x1el\xdc\x9f\xf8+]-\r\x17\x85\xb2\x8fG\xb1H\x12\x99\xcd\x12\xee\xd9\xc0L\xa4\xd2\xc6\xec\xff<\x1c|im\x84\xf3JdF\x99\xe9l\x00\x0f\xf5\xfe\x8f\xeb\xef_\xe6\xff\x19+9\xf1\x9a8\xdf!\xfc\xfan\x9bk\xdff1\x07|kk\xdf\x8c!:\x9b\xc1\x1d?t\xdb\xcc\xa7v\xdank\x12\xefx\xed\xebm\xa5}\xb6\xb1\xb7\xa2\xbc\x9

In [ ]:
!unzip train_data.zip


Archive:  train_data.zip
replace data/train_tokens.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
!pip install transformers datasets torch accelerate


In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))


Embedding(50257, 768)

In [ ]:
from datasets import Dataset

with open("data/train_tokens.txt") as f:
    lines = [l.strip() for l in f if l.strip()]

dataset = Dataset.from_dict({"text": lines})


In [ ]:
def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

tokenized_ds = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])


Map:   0%|          | 0/1047 [00:00<?, ? examples/s]

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./waf_gpt2",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
    fp16=True,
    report_to="none"
)


In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # IMPORTANT: GPT-2 is NOT masked LM
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    data_collator=data_collator
)

trainer.train()


Step,Training Loss
100,0.845600


TrainOutput(global_step=198, training_loss=0.5337058462277807, metrics={'train_runtime': 54.8932, 'train_samples_per_second': 57.22, 'train_steps_per_second': 3.607, 'total_flos': 102589784064000.0, 'train_loss': 0.5337058462277807, 'epoch': 3.0})

In [ ]:
trainer.save_model("waf_gpt2")
tokenizer.save_pretrained("waf_gpt2")


('waf_gpt2/tokenizer_config.json',
 'waf_gpt2/special_tokens_map.json',
 'waf_gpt2/vocab.json',
 'waf_gpt2/merges.txt',
 'waf_gpt2/added_tokens.json')

In [ ]:
!zip -r waf_gpt2.zip waf_gpt2


updating: waf_gpt2/ (stored 0%)
updating: waf_gpt2/special_tokens_map.json (deflated 74%)
updating: waf_gpt2/generation_config.json (deflated 24%)
updating: waf_gpt2/vocab.json (deflated 68%)
updating: waf_gpt2/model.safetensors (deflated 7%)
updating: waf_gpt2/training_args.bin (deflated 53%)
updating: waf_gpt2/checkpoint-198/ (stored 0%)
updating: waf_gpt2/checkpoint-198/special_tokens_map.json (deflated 74%)
updating: waf_gpt2/checkpoint-198/generation_config.json (deflated 24%)
updating: waf_gpt2/checkpoint-198/vocab.json (deflated 68%)
updating: waf_gpt2/checkpoint-198/model.safetensors (deflated 7%)
updating: waf_gpt2/checkpoint-198/training_args.bin (deflated 53%)
updating: waf_gpt2/checkpoint-198/scaler.pt (deflated 64%)
updating: waf_gpt2/checkpoint-198/trainer_state.json (deflated 55%)
updating: waf_gpt2/checkpoint-198/config.json (deflated 51%)
updating: waf_gpt2/checkpoint-198/rng_state.pth (deflated 26%)
updating: waf_gpt2/checkpoint-198/optimizer.pt (deflated 8%)
updating

In [ ]:
files.download("waf_gpt2.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
prompt = "method_GET path_ / rest / products"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_length=30,
    do_sample=True,
    top_k=50
)

print(tokenizer.decode(outputs[0]))


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


method_GET path_ / rest / products QUERY = False QUERY = False QUERY_ = False QUERY = False QUERY_GET


In [ ]:
import torch
import math

def compute_perplexity(text):
    encodings = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=64
    )

    encodings = {k: v.to(model.device) for k, v in encodings.items()}

    with torch.no_grad():
        outputs = model(**encodings, labels=encodings["input_ids"])
        loss = outputs.loss

    return math.exp(loss.item())


In [ ]:
benign = "method_GET path_ / rest / products QUERY = False"
print("Benign perplexity:", compute_perplexity(benign))


Benign perplexity: 1.14840591552335


In [ ]:
malicious = "method_GET path_ / rest / products / search ? q = ' OR 1 = 1 --"
print("Malicious perplexity:", compute_perplexity(malicious))


Malicious perplexity: 20.40324390111821


In [ ]:
import random

samples = random.sample(lines, min(100, len(lines)))
ppls = [compute_perplexity(s) for s in samples]

print("Max benign perplexity:", max(ppls))
print("Mean benign perplexity:", sum(ppls)/len(ppls))


Max benign perplexity: 2.660686495047744
Mean benign perplexity: 1.260703202992895


In [ ]:
# threshold = max_benign * 1.5